**Link Github UI Page:**
[Github Page](https://detmaneh.github.io/nyc-taxi-data-profiling-uacj/#top)

**Link Github Repository:**
[Github Repository](https://github.com/detmaneh/nyc-taxi-data-profiling-uacj)

**Instalamos lo necesario para correr la actividad**


**Cargar el dataset en un DataFrame de PySpark.**

In [1]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

from google.colab import drive
drive.mount('/content/drive')
# Obtenemos el csv y creamos una sesion
spark = SparkSession.builder.appName("Practica_NSL_KDD").getOrCreate()
print("Sesion Creada")

Mounted at /content/drive
Sesion Creada


**Mostrar:**

*   Número total de filas.
*   Número total de columnas.
*   Primeras 10 filas utilizando show().



In [2]:
ruta_archivo = "/content/drive/MyDrive/NSL_KDD.csv" # Documento que necesitamos para la actividad

# Leemos los datos de el documento
df = spark.read.csv(ruta_archivo, header=True, inferSchema=True)

total_filas = df.count()
total_columnas = len(df.columns)
# Nos Aseguramos de que se lee correctamente o por lo menos sin errores
print(f"\nFilas: {total_filas}")
print(f"\nColumnas: {total_columnas}")
print("\nSchema:")
df.printSchema()
# Hacemos peek a nuestra estructura solo 10 elementos
print("\nDataset Peek:")
df.show(10)


Filas: 18035

Columnas: 42

Schema:
root
 |-- duration: string (nullable = true)
 |-- protocol_type: string (nullable = true)
 |-- service: string (nullable = true)
 |-- flag: integer (nullable = true)
 |-- src_bytes: integer (nullable = true)
 |-- dst_bytes: integer (nullable = true)
 |-- land: integer (nullable = true)
 |-- wrong_fragment: integer (nullable = true)
 |-- urgent: integer (nullable = true)
 |-- hot: integer (nullable = true)
 |-- num_failed_logins: integer (nullable = true)
 |-- logged_in: integer (nullable = true)
 |-- num_compromised: integer (nullable = true)
 |-- root_shell: integer (nullable = true)
 |-- su_attempted: integer (nullable = true)
 |-- num_root: integer (nullable = true)
 |-- num_file_creations: integer (nullable = true)
 |-- num_shells: integer (nullable = true)
 |-- num_access_files: integer (nullable = true)
 |-- num_outbound_cmds: integer (nullable = true)
 |-- is_host_login: integer (nullable = true)
 |-- is_guest_login: integer (nullable = true)

**Seleccionar únicamente las siguientes columnas utilizando select():**

*   protocol_type
*   service
*   src_bytes
*   dst_bytes
*   label



In [3]:
# Seleccionamos las columnas de la actividad
try:
  df = df.select("protocol_type", "service", "src_bytes", "dst_bytes", "label")
except Exception as e:
  print("\nError con las columnas DESFAZADAS:")
  print(e)
  print("Utilizamos las columnas originales renombrando el error con la recomendacion del error original")
  # Las columnas originales provenientes del documento de la actividad
  columnas_actuales = df.columns
  # Recorremos hacia la izquierda: segunda columna (índice 1)
  # y hasta el final, y agregamos 'label' al final
  columnas_recorridas = columnas_actuales[1:] + ['label']
  # Cambiamos las columnas de nuestro Data Frame
  df_recorrido = df.toDF(*columnas_recorridas)
  df = df_recorrido.select("protocol_type", "service", "src_bytes", "dst_bytes", "label")
print("\nShow 5 de nuestro Dataframe")
df.show(5)

{"ts": "2026-05-16 15:36:17.221", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `label` cannot be resolved. Did you mean one of the following? [`land`, `class`, `flag`, `hot`, `count`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o33.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `label` cannot be resolved. Did you mean one of the following? [`land`, `class`, `flag`, `hot`, `count`]. SQLSTATE: 42703;\n'Project [protocol_type#18, service#19, src_bytes#21, dst_bytes#22, 'label]\n+- Relation [duration#17,protocol_type#18,service#19,flag#20,src_bytes#21,dst_


Error con las columnas DESFAZADAS:
[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `label` cannot be resolved. Did you mean one of the following? [`land`, `class`, `flag`, `hot`, `count`]. SQLSTATE: 42703;
'Project [protocol_type#18, service#19, src_bytes#21, dst_bytes#22, 'label]
+- Relation [duration#17,protocol_type#18,service#19,flag#20,src_bytes#21,dst_bytes#22,land#23,wrong_fragment#24,urgent#25,hot#26,num_failed_logins#27,logged_in#28,num_compromised#29,root_shell#30,su_attempted#31,num_root#32,num_file_creations#33,num_shells#34,num_access_files#35,num_outbound_cmds#36,is_host_login#37,is_guest_login#38,count#39,srv_count#40,serror_rate#41,... 17 more fields] csv

Utilizamos las columnas originales renombrando el error con la recomendacion del error original

Show 5 de nuestro Dataframe
+-------------+-------+---------+---------+-----+
|protocol_type|service|src_bytes|dst_bytes|label|
+-------------+-------+---------+---------+-----+
|  

**Crear una nueva columna llamada:**

total_bytes y contendra **src_bytes + dst_bytes**

In [5]:
# A nuestro Data Frame filtrado le agregamos una columna,
# y le sumamos los otros bytes del renglon al final de cada hilera
df = df.withColumn("total_bytes", col("src_bytes") + col("dst_bytes"))
df.summary().show()

+-------+-------------+-------+------------------+------------------+------------------+------------------+
|summary|protocol_type|service|         src_bytes|         dst_bytes|             label|       total_bytes|
+-------+-------------+-------+------------------+------------------+------------------+------------------+
|  count|        18035|  18035|             18035|             18035|             18035|             18035|
|   mean|         NULL|   NULL|11319.993734405323|2184.3236484613253|18.018519545328527|13504.317382866648|
| stddev|         NULL|   NULL| 527492.8189842424|23471.497691741282| 4.269036942356893| 529284.9463205778|
|    min|         icmp|    IRC|                 0|                 0|                 0|                 0|
|    25%|         NULL|   NULL|                 0|                 0|                17|                 0|
|    50%|         NULL|   NULL|                54|                46|                20|               180|
|    75%|         NULL|   NU

**Ordenar los registros de mayor a menor utilizando la columna:**

In [6]:
print("Ordenamos todas las hileras de nuestro dataframe por total_bytes de mayor a menor")
df = df.orderBy(col("total_bytes").desc())
df.summary().show()

Ordenamos todas las hileras de nuestro dataframe por total_bytes de mayor a menor
+-------+-------------+-------+------------------+------------------+------------------+------------------+
|summary|protocol_type|service|         src_bytes|         dst_bytes|             label|       total_bytes|
+-------+-------------+-------+------------------+------------------+------------------+------------------+
|  count|        18035|  18035|             18035|             18035|             18035|             18035|
|   mean|         NULL|   NULL|11319.993734405323|2184.3236484613253|18.018519545328527|13504.317382866648|
| stddev|         NULL|   NULL| 527492.8189842422|23471.497691741126| 4.269036942356909| 529284.9463205736|
|    min|         icmp|    IRC|                 0|                 0|                 0|                 0|
|    25%|         NULL|   NULL|                 0|                 0|                17|                 0|
|    50%|         NULL|   NULL|                54|    

**Mostrar los valores únicos de:**

In [7]:
print("Valores únicos basandonos en la columna 'protocol_type':")
df.select("protocol_type").distinct().show()

Valores únicos basandonos en la columna 'protocol_type':
+-------------+
|protocol_type|
+-------------+
|          tcp|
|          udp|
|         icmp|
+-------------+



**Agrupar los registros utilizando:**

In [8]:
print("Agrupamos todo por el mismo valor en la columna de protocol_type")
print("\nObtenemos el conteo de matches con valores iguales de protocol_type")
df.groupBy("protocol_type").count().show()

Agrupamos todo por el mismo valor en la columna de protocol_type

Obtenemos el conteo de matches con valores iguales de protocol_type
+-------------+-----+
|protocol_type|count|
+-------------+-----+
|          tcp|15136|
|          udp| 2074|
|         icmp|  825|
+-------------+-----+



**Agrupar por:**

In [9]:
print("Agrupar por la columna label y de eso calculamos el promedio de src_bytes")
df.groupBy("label").avg("src_bytes").show()

Agrupar por la columna label y de eso calculamos el promedio de src_bytes
+-----+------------------+
|label|    avg(src_bytes)|
+-----+------------------+
|   12|  9763.74293059126|
|    1| 6299.420289855072|
|   13|  3148.15421686747|
|   16| 23404.57981651376|
|    6|13574.396825396825|
|    3| 65704.45161290323|
|   20| 8169.303538175047|
|    5|14005.048780487805|
|   19|16109.481741573034|
|   15|20715.175345377258|
|    9|15906.785714285714|
|   17|5141.9304812834225|
|    4|1204952.5308641975|
|    8|14678.409523809523|
|    7|          12820.58|
|   10| 8000.961538461538|
|   21| 350.5807130333138|
|   11| 6856.350543478261|
|   14|15336.081494057726|
|    2| 45254.47727272727|
+-----+------------------+
only showing top 20 rows


En resumen, al usar groupBy nos permite saber qué es lo que tiene la tabla, y en un futuro nos podría ayudar a identificar si hay algo fuera de serie, o no esperado.
Cuando sacamos el promedio de src_bytes nos ayuda a saber qué tanto tráfico existe, se puede ordenar de la forma que queramos, pero es una fuente muy buena para saber el promedio de volumen de datos por label.
También cuando filtramos por tcp y mayor a 1000, yo me imagino que se podría usar para ver usos externos del sistema, si hay algo raro se puede detectar de dónde viene y así investigar más a fondo si hay algún agente “malo” usándolo.
Por ultimo, lo mas importante fue que la tabla estaba desfasada, y como el ejercicio requería usar label, pero no había label, y otras columnas se veían mal, como datos no correspondientes a el header, los movi para que se pudiera usar la información, y cada columna por lo menos a mi parecer tuvieran el header correcto.